# Naturalistic EMA prediction based on ACC - Dyskinesia use case

## 0) Imports

- document versions for reproducibility

In [ ]:
# import packages
import datetime as dt
import pandas as pd
import numpy as np
import os
import sys
import csv
import json
import importlib
from itertools import product, compress
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, variation

from dataclasses import dataclass, field


In [ ]:
print('Python sys', sys.version)
print('pandas', pd.__version__)
print('numpy', np.__version__)
# print('mne_bids', mne_bids.__version__)
# print('mne', mne.__version__)
# print('sci-py', scipy.__version__)
# print('sci-kit learn', sk.__version__)
# print('matplotlib', plt_version)

"""
Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.1.1
numpy 1.26.0

from 16.09

Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.3.2
numpy 2.3.3
"""

Import custom functions

In [ ]:
# from dbs_home repo
from dbs_home.load_raw.main_load_raw import loadSubject 
import dbs_home.utils.helpers as home_helpers
import dbs_home.utils.ema_utils as home_ema_utils
import dbs_home.plot_data.plot_compliance as plot_home_compl
import dbs_home.preprocessing.preparing_ema as home_ema_prep

from dbs_home.preprocessing import get_submovements
import dbs_home.preprocessing.acc_preprocessing as acc_prep
import dbs_home.preprocessing.submovement_processing as submove_proc
import dbs_home.load_raw.load_watch_raw as load_watch


In [ ]:
# from current repo
from utils import load_utils, load_data, prep_data
import utils.acc_features as acc_fts
import utils.ft_extract_class as ft_extr_class
import utils.feat_extraction as ft_extr
import utils.data_handling_ema_acc as data_handling

from plotting import plot_help



## 1) Load and check naturalistic data

In [ ]:
# set paths

feas_data_path = os.path.join(
    os.path.dirname(load_utils.get_onedrive_path()),
    'PROJECTS', 'home_feasibility'
)
feas_fig_path = os.path.join(
    load_utils.get_onedrive_path('figures'),
    'feasibility'
)



#### Load ACC data
- create SVM
- filter data within the dataclass

In [ ]:
# LID
sub_id = 'hm24'
ses_id = 'ses03'

FT_PARAMS_VERSION = 'v3'  # v3 updated lid version

In [ ]:
# import naturalistic data via dbs_home repo




### test days for hm24-ses01  # dyskinesia
# dev_day_selection = ['2025-07-17', '2025-07-18']
# dev_day_selection = [f'2025-07-{d}' for d in np.arange(17, 31)]
dev_day_selection = []

### test days for hm20-ses01  # tremor
# dev_day_selection = [
#     '2025-06-13', '2025-06-14',
#     '2025-06-15', '2025-06-16'
# ]

home_dat = loadSubject(
    sub=sub_id,
    ses=ses_id,
    incl_STEPS=False,
    incl_EPHYS=False,
    incl_EMA=True,
    incl_ACC=True,
    day_selection=dev_day_selection
)

Check available EMAs

In [ ]:
plot_home_compl.plot_EMA_completion_perSession(home_dat)

## 2) ACC processing incl. Feature extraction

#### Extract features from Acc-Windows aligned to EMAs

full day extraction and visualization at end of notebook

In [ ]:
importlib.reload(ft_extr_class)
importlib.reload(acc_fts)
importlib.reload(data_handling)
importlib.reload(ft_extr)


xy_dict = {}

# iter_settings = {
#     'nosm_allwindow':[False, True, True],
#     'sm_merged': [True, False, False],
#     'sm_single': [True, False, True]}

ft_settings = {
    'nosm_allwindow': {
        'EXTRACT_FT_FROM_SMs': False,
        'EXTRACT_FT_FULL_WIN': True,
        'ACC_FEATS_on_SINGLE_MOVES': True
    },
    'sm_merged': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': False
    },
    'sm_single': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': True
    }
}




for ft_type, ft_settings in ft_settings.items():

    xy_dict[ft_type] = ft_extr.get_features_per_session(
        home_dat=home_dat,
        sub_id=sub_id,
        ses_id=ses_id,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        LOAD_SAVE_FEATS = True,
        # define how features should be extracted
        STORE_SUBMOVES = False,
        # plotting settings
        SAVE_PLOT = False,
        SHOW_PLOT = False,
        **ft_settings
    )

In [ ]:
[f'{k}: {v.shape}, column-names: {list(v.keys())}' for k, v in xy_dict.items()]

## 3a) Evaluate extracted Features

- Hssayeni et al, Scientific Reports 2021
    - strongest wrist-features: angular velocity, standard deviation, power of secondary frequency, power of 1–4 Hz band, and Shannon Entropy (r = 0.82  - r = 0.75)

- from svm: classic features
- include cross-corr between pc1 and pc2


In [ ]:
from plotting import plot_home_preds as plt_preds
import plotting.plot_ft_explor as plt_fts

define selected data

In [ ]:
SES = 'ses01'
FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single
FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
EMA_ref = 'LID'



In [ ]:
importlib.reload(ft_extr)


xy_dict = {}

for ses_id in ['ses01', 'ses02', 'ses03']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=True,
    )
    xy_dict[ses_id] = tempdf

In [ ]:
ft_df = xy_dict['ses01'].copy()


check selected variables in dataframe and first rows of dataframe

In [ ]:
print(ft_df.shape)
print(ft_df.keys())
print(ft_df.head())
print([f'LID "{k}": {v} counts' for k, v in zip(
    *np.unique(ft_df[EMA_ref], return_counts=True))])


In [ ]:
importlib.reload(ft_extr)

# define which features to include in prediction and which to exclude (times, ema)
feats_incl, EMA_CODING = ft_extr.get_feat_params(FT_PARAMS_VERSION)
keys_excl = ['timestamp'] + list(EMA_CODING.keys())  # times + ema

Visualize feature distributions

In [ ]:
importlib.reload(plt_fts)

save_plot = False


for ft_name in ft_df.keys():
    print(f'\n{ft_name}')
    # if ft_name not in ft_df.keys():
    #     print(f'Feature {ft_name} not found in dataframe columns. Skipping.')
    #     continue
    if ft_name in keys_excl:
        print(f'Feature {ft_name} is not included in the feature set. Skipping.')
        continue
    
    ### plot distributions with histo, scatter, and boxes
    plt_fts.plot_ft_distribution(
        ft_df=ft_df,
        ft_name=ft_name,
        EMA_ref=EMA_ref,
        sub_id=sub_id,
        SES=SES,
        FT_TYPE=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        save_plot=save_plot,
    )



## 3b) Predict EMAs based on Wearable-features

#### LID-classification: cross-validation based on preop session

- only EMA windows
- cv-folds: n-folds of EMA-windows from all sessions

In [ ]:
import utils.pred_utils as pred_utils

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, LogisticRegression

from sklearn.metrics import (
    r2_score, root_mean_squared_error,
    mean_absolute_error, cohen_kappa_score
)
from scipy.stats import spearmanr, kendalltau, mannwhitneyu


In [ ]:
# define window for cross-validation
cv_df = xy_dict['ses01'].copy()
print(cv_df.keys())

FT_TYPE = 'sm_merged'    # nosm_allwindow, sm_merged, sm_single
FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
EMA_Y = 'LID'

EXCL_HR = False  # exclude heart rate features (not available in all timepoints)
LOG_SKEWED = True
TRANSFORM_Y = False  # transform LID into 3 classes: 1, 2 (1-4), 3 (>4)

PRED_FTS = pred_utils.get_keys_incl(cv_df.keys(), excl_hr=EXCL_HR,)

print(f'model trained on features: {PRED_FTS}')

In [ ]:
importlib.reload(pred_utils)

### define training X, y
X_all = cv_df[PRED_FTS].values.copy()
y_all = cv_df[EMA_Y].values.copy().astype(float)

### prepare data
skew_corr_bool_list = [pred_utils.check_skewness(
    X_all[:, i][~np.isnan(X_all[:, i])],
    threshold=1.5,
)[0] for i in range(X_all.shape[1])]
X_all, y_all = pred_utils.prepare_Xy_for_regression(
    X_all, y_all,
    remove_nan_rows=True,
    transform_skewed_feats=LOG_SKEWED,
    verbose=True,
)

In [ ]:
importlib.reload(pred_utils)

# pipe includes scaling based on training data, and regression model
cv_pipe = Pipeline([
    # ('corr_filter', pred_utils.DropCorrelatedFeatures(threshold=0.9)),
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
#     # ('reg', LogisticRegression(multi_class='multinomial',)),
])

y_pred_all = np.zeros_like(y_all)

n_folds = 4
cv_folds = KFold(n_splits=n_folds).split(X_all)

for i_fold, (train_idx, test_idx) in enumerate(cv_folds):
    print(f'test indices for fold {i_fold + 1}: {test_idx}')
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train, y_test = y_all[train_idx], y_all[test_idx]

    # train regression model
    cv_pipe.fit(X_train, y_train)
    # predict test set (incl scaling of fts based on train set)
    y_pred = cv_pipe.predict(X_test)
    # clip predictions to valid range and round to nearest integer
    y_pred = np.clip(np.round(y_pred), y_train.min(), y_train.max()).astype(int)
    y_pred_all[test_idx] = y_pred  # add predictions to total array in correct indices
    
    print(f'finished fold {i_fold + 1} / {n_folds}')

# autocorr_col_idx = [1, 4, 5, 6, 11, 14, 19, 20, 21, 25]
# print(f'feats often removed: {np.array(PRED_FTS)[autocorr_col_idx]}')


In [ ]:
if (FT_TYPE == 'sm_merged' and FT_PARAMS_VERSION == 'v3' and EXCL_HR == False
    and LOG_SKEWED == True):
    autocorr_drop_col_idx = [1, 4, 5, 6, 11, 14, 19, 20, 21, 25]
elif (FT_TYPE == 'sm_merged' and FT_PARAMS_VERSION == 'v3' and EXCL_HR == False
    and LOG_SKEWED == False):
    autocorr_drop_col_idx = [1, 4, 5, 6, 11, 14, 19, 21, 28]

print(f'selected drop col idx: {autocorr_drop_col_idx}')

quick evaluation, check sign with permutations

In [ ]:
rho, p = spearmanr(y_all, y_pred_all)
print(f"Spearman rho={rho.round(2)}, p={p.round(5)}")

tau, p_tau = kendalltau(y_all, y_pred_all)
print(f"Kendall tau={tau.round(2)}, p={p_tau.round(5)}")  

mae = mean_absolute_error(y_all, y_pred_all)
kappa = cohen_kappa_score(y_all, y_pred_all, weights='linear')
print(f"Mean Absolute Error: {mae:.2f}, Cohen's Kappa: {kappa:.2f}")

r2 = r2_score(y_all, y_pred_all)
rmse = root_mean_squared_error(y_all, y_pred_all)
print(f"### R2 for all data: {r2:.2f}, RMSE: {rmse:.2f}")

cv_results = {
    'spearman_rho': rho,
    'spearman_p': p,
    'kendall_tau': tau,
    'kendall_p': p_tau,
    'mae': mae,
    'kappa': kappa,
    'r2': r2,
    'rmse': rmse
}


fig, ax = plt.subplots(1, 1, figsize=(6, 6))

y_jitter = np.random.normal(0, 0.1, size=y_all.shape)  # add jitter for better visibility
ax.scatter(y_all + y_jitter, y_pred_all, alpha=0.5)
ax.set_xlabel('True LID')
ax.set_ylabel('Predicted LID')

plt.show()

## 3c) Test LID classifier on post-op sessions

- Train-split: EMA-windows ses01, Test-split: EMA-windows ses02 and ses03

Prepare cv-training data and fit model

In [ ]:
traindf = ft_extr.get_feat_df_for_pred(
    sub_id=sub_id,
    ses_id=ses_id,
    ft_set_sel=FT_TYPE,
    FT_PARAMS_VERSION=FT_PARAMS_VERSION,
    ONLY_EMA_WINDOWS=True,
)
pred_fts = pred_utils.get_keys_incl(traindf.keys(), excl_hr=EXCL_HR,)


for i, ses_id in enumerate([ 'ses03']):  # 'ses02',
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=True,
    )
    if i == 0: testdf = tempdf.copy()
    else: testdf = pd.concat([testdf, tempdf],).reset_index(drop=True)

print(f'testdf shape: {testdf.shape}, columns: {testdf.keys()}')

In [ ]:
# pipe includes scaling based on training data, and regression model
test_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
    # ('reg', LogisticRegression(multi_class='multinomial',)),
])

### prepare train and test data

incl_fts = pred_fts.copy()

# excl autocorrelating features based on cv-training data
# print(f'features to be excluded based on autocorrelation in training data: '
#       f'{np.array(pred_fts)[autocorr_drop_col_idx]}')
# incl_fts = [f for i, f in enumerate(incl_fts) if i not in autocorr_drop_col_idx]

X_train = xy_dict['ses01'][incl_fts].values.copy()
y_train = xy_dict['ses01'][EMA_Y].values.copy().astype(float)


### prepare data

# check which features will be log-transformed in training
if LOG_SKEWED:
    skew_corr_bool_list = [pred_utils.check_skewness(
        X_train[:, i][~np.isnan(X_train[:, i])],
        threshold=1.5,
    )[0] for i in range(X_train.shape[1])]

X_train, y_train = pred_utils.prepare_Xy_for_regression(
    X_train, y_train,
    remove_nan_rows=True,
    transform_skewed_feats=LOG_SKEWED,
    verbose=True,
)

# train regression model
test_pipe.fit(X_train, y_train)

prepare test data and predict

In [ ]:
testdf.shape

In [ ]:

# prepare test data
X_test = testdf[incl_fts].values.copy()
y_test = testdf[EMA_Y].values.copy().astype(float)

X_test, y_test = pred_utils.prepare_Xy_for_regression(
    X_test, y_test,
    remove_nan_rows=True,
    transform_skewed_feats=False,  # always independent logging outside of this function for test data
    verbose=True,
)

# only log-transform test-features that were log-transformed in training
if LOG_SKEWED:
    for i in range(X_test.shape[1]):
        if skew_corr_bool_list[i]:  # if this feature was log-transformed in training
            X_test[:, i] = pred_utils.log_transf(X_test[:, i])


# predict test set (incl scaling of fts based on train set)
y_pred = test_pipe.predict(X_test)
# clip predictions to valid range and round to nearest integer
y_pred = np.clip(np.round(y_pred), y_train.min(), y_train.max()).astype(int)

print(f'present answer in true LID data: {np.unique(y_test, return_counts=True)}')

validation test of true predictions

In [ ]:
rho, p = spearmanr(y_test, y_pred)
print(f"Spearman rho={rho:.2f}, p={p:.5f}")

tau, p_tau = kendalltau(y_test, y_pred)
print(f"Kendall tau={tau:.2f}, p={p_tau:.5f}")  

mae = mean_absolute_error(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred, weights='linear')
print(f"Mean Absolute Error: {mae:.2f}, Cohen's Kappa: {kappa:.2f}")

r2 = r2_score(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
print(f"### R2 for all data: {r2:.2f}, RMSE: {rmse:.2f}")


test_results = {
    'spearman_rho': rho,
    'spearman_p': p,
    'kendall_tau': tau,
    'kendall_p': p_tau,
    'mae': mae,
    'kappa': kappa,
    'r2': r2,
    'rmse': rmse
}


fig, ax = plt.subplots(1, 1, figsize=(6, 6))

# y_jitter = np.random.normal(0, 0.1, size=y_test.shape)  # add jitter for better visibility
# ax.scatter(y_test + y_jitter, y_pred + y_jitter, alpha=0.5)

# boxplots of predicted value per true LID class
data_to_plot = [y_pred[y_test == lid] for lid in np.unique(y_test)]
ax.boxplot(data_to_plot, labels=np.unique(y_test))

ax.set_xlabel('True LID')
ax.set_ylabel('Predicted LID')

plt.show()

try permutations for current data

In [ ]:
perm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
    # ('reg', LogisticRegression(multi_class='multinomial',)),
])

n_perm = 1000
np.random.seed(27)  # for reproducibility

perm_results = {
    'rho': [],
    'tau': [],
    'mae': [],
    'kappa': [],
    'r2': [],
    'rmse': []
}

for i in range(n_perm):
    print(f'Permutation {i + 1} / {n_perm}')
    y_perm_train = y_train.copy()
    np.random.shuffle(y_perm_train)

    # take random n-samples of values 1 to 9, with free duplication
    # y_perm_train = np.random.choice(
    #     np.arange(1, 10), size=y_train.shape[0], replace=True,
    # )

    # train regression model with permuted training labels
    perm_pipe.fit(X_train, y_perm_train)
    # predict test set (incl scaling of fts based on train set)
    y_pred_perm = perm_pipe.predict(X_test)
    # clip predictions to valid range and round to nearest integer
    y_pred_perm = np.clip(np.round(y_pred_perm), y_train.min(), y_train.max()).astype(int)

    # calculate metrics for permuted labels
    perm_results['rho'].append(spearmanr(y_test, y_pred_perm)[0])
    perm_results['tau'].append(kendalltau(y_test, y_pred_perm)[0])
    perm_results['mae'].append(mean_absolute_error(y_test, y_pred_perm))
    perm_results['kappa'].append(cohen_kappa_score(y_test, y_pred_perm, weights='linear'))
    perm_results['r2'].append(r2_score(y_test, y_pred_perm))
    perm_results['rmse'].append(root_mean_squared_error(y_test, y_pred_perm))
    

In [ ]:
# plot histograms of permuted metrics with observed metric as vertical line

incl_metrics = ['mae', 'rmse']

fig, axes = plt.subplots(1, len(incl_metrics), figsize=(5 * len(incl_metrics), 5))

for i, metric in enumerate(incl_metrics):

    # calculate p-value as proportion of permuted metrics that are as extreme as or more extreme than the observed metric
    if metric in ['rho', 'tau', 'kappa', 'r2']:  # for metrics where higher is better, p-value is proportion of permuted metrics >= observed metric
        p_value = np.mean(np.array(perm_results[metric]) >= test_results[metric])
    else:  # for metrics where lower is better, p-value is proportion of permuted metrics <= observed metric
        p_value = np.mean(np.array(perm_results[metric]) <= test_results[metric])
    print(f'{metric}: observed={test_results[metric]:.2f}, p-value={p_value:.4f}')

    ax = axes[i]
    ax.hist(perm_results[metric], bins=30,
            color='blue', alpha=.5, label='Permuted',)
    obs_metric = test_results[metric]  # get observed metric value from variable
    ax.axvline(obs_metric, color='red', linestyle='--', label=f'Observed {metric}={obs_metric:.2f}')
    ax.set_title(f'Permutation distribution of {metric}')
    ax.set_xlabel(metric)
    ax.set_ylabel('Frequency')
    ax.legend()
    
plt.tight_layout()
plt.show()

## 4) Full day application

### 4a) Data prep
- feature extractions for full days
- create one data structure
- TODO: test nosubmove_features for ses01 and ses03





Import fullday test data

In [ ]:
lid_preds_figpath = os.path.join(load_utils.get_onedrive_path('figures'), 'lid_usecase_preds')

In [ ]:
## define parameters for full day testing 
sub_id = 'hm24'
FT_TYPE = 'sm_merged'
FT_PARAMS_VERSION = 'v3'  # v3 updated lid version
EXCL_HR = False  # exclude heart rate features (not available in all timepoints)
LOG_SKEWED = True

In [ ]:
importlib.reload(ft_extr)

fullday_dfs = {}

for ses_id in ['ses01',  'ses03']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel = FT_TYPE,
        ONLY_EMA_WINDOWS=False,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
    )
    pred_keys = pred_utils.get_keys_incl(tempdf.keys(), excl_hr=EXCL_HR,)
    fullday_dfs[ses_id] = tempdf[pred_keys]

Import EMA-linked pre-op training data, and fit model

In [ ]:
# define pipeline for training on EMA-ses01, and testing on full days
fullday_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', Ridge(alpha=1.0)),
])

fit_df = ft_extr.get_feat_df_for_pred(
    sub_id=sub_id,
    ses_id='ses01',
    ft_set_sel=FT_TYPE,
    FT_PARAMS_VERSION=FT_PARAMS_VERSION,
    ONLY_EMA_WINDOWS=True,
)
fit_X = fit_df[pred_keys].values.copy()
fit_y = fit_df[EMA_Y].values.copy().astype(float)

if LOG_SKEWED:
    skew_corr_bool_list = [pred_utils.check_skewness(
        fit_X[:, i][~np.isnan(fit_X[:, i])],
        threshold=1.5,
    )[0] for i in range(fit_X.shape[1])]

fit_X, fit_y = pred_utils.prepare_Xy_for_regression(
    fit_X, fit_y,
    remove_nan_rows=True,
    transform_skewed_feats=LOG_SKEWED,
    verbose=True,
)

# fit the pipeline on the training data
fullday_pipe.fit(fit_X, fit_y)

### 4b) Apply trained model on full day data


In [ ]:
# print(fullday_dfs['ses01'].shape, fullday_dfs['ses02'].shape, fullday_dfs['ses03'].shape)
fullday_dfs['ses01'].head()

In [ ]:
session_code = np.concatenate(
    [[ses_id] * len(fullday_dfs[ses_id]) for ses_id in fullday_dfs.keys()]
)
times_full = np.concatenate(
    [fullday_dfs[ses_id].index.values for ses_id in fullday_dfs.keys()]
)
X_full = np.concatenate(
    [fullday_dfs[ses_id].values for ses_id in fullday_dfs.keys()]
)
assert len(times_full) == X_full.shape[0] == len(session_code), (
    "Number of timestamps does not match number of data rows."
)

print(f'all data shape: {X_full.shape}, time range: {times_full.min()} - {times_full.max()}')
print(f'session codes: {np.unique(session_code, return_counts=True)}')


# no y while no true labels
X_full, nan_rows = pred_utils.prepare_Xy_for_regression(
    X_full,
    remove_nan_rows=True,
    return_nanrow_bool=True,
    transform_skewed_feats=False,  #  for test data, always independent based on training-skewlogging
    verbose=True,
)
session_code = session_code[~nan_rows]  # remove session codes for rows with NaN values in features
times_full = times_full[~nan_rows]  # remove timestamps for rows with NaN

if LOG_SKEWED:
    for i in range(X_full.shape[1]):
        if skew_corr_bool_list[i]:  # if this feature was log-transformed in training
            X_full[:, i] = pred_utils.log_transf(X_full[:, i])

# predict test set (incl scaling of fts based on train set)
y_full_pred = fullday_pipe.predict(X_full)
# clip predictions to valid range and round to nearest integer
y_full_pred = np.clip(np.round(y_full_pred), 1, 9).astype(int)

### 4c) Evaluate, visualize full day results
- overall values preop vs postop
- differences in day-time-blocks
- differences around med-intakes

In [ ]:
from plotting import plot_home_preds as plt_preds

In [ ]:
SES_COLORS = {'ses01': 'violet', 'ses02': 'orange', 'ses03': 'darkcyan'}


In [ ]:
# statistically compare predicted LID distributions between sessions
if len(np.unique(session_code)) == 2:  # only perform test if there are exactly 2 sessions to compare
      ses_groups = [y_full_pred[session_code == ses_id] for ses_id in np.unique(session_code)]
      S, p = mannwhitneyu(ses_groups[0], ses_groups[1])

      # describe session groups
      for ses_id, group in zip(np.unique(session_code), ses_groups):
            print(f'session {ses_id}: n={len(group)}, '
                  f'mean predicted LID={group.mean():.2f}, std={group.std():.2f}')

      print(f'pre-DBS vs post-DBS: mwu-stat = {S.round(2)}, p = {p.round(4)}')
      print(f'data settings: FT_TYPE={FT_TYPE}, EXCL_HR={EXCL_HR}, '
            f'LOG_SKEWED={LOG_SKEWED}, ft_version={FT_PARAMS_VERSION}')

elif len(np.unique(session_code)) == 3:
      for ses_id in ['ses02', 'ses03']:
            print(f'\nComparing ses01 vs {ses_id}')
            ses_groups = [y_full_pred[session_code == 'ses01'],
                          y_full_pred[session_code == ses_id]]
            S, p = mannwhitneyu(ses_groups[0], ses_groups[1])

            print(f'session {ses_id}: n={len(group)}, '
                  f'mean predicted LID={group.mean():.2f}, std={group.std():.2f}')
            print(f'group sizes: {[len(g) for g in ses_groups]}')
            print(f'pre-DBS vs post-DBS: mwu-stat = {S.round(2)}, p = {p.round(4)}')
            print(f'data settings: FT_TYPE={FT_TYPE}, EXCL_HR={EXCL_HR}, '
                  f'LOG_SKEWED={LOG_SKEWED}, ft_version={FT_PARAMS_VERSION}')

In [ ]:
FT_TYPE, EXCL_HR, LOG_SKEWED

In [ ]:
importlib.reload(data_handling)
importlib.reload(plt_preds)

save_plot = False
figname = f'fullSession_preds_LID_{FT_TYPE}_exclHR-{EXCL_HR}_logSkew-{LOG_SKEWED}_ftV-{FT_PARAMS_VERSION}.png'

fig, axes = plt.subplots(1, 2, figsize=(18, 6),
                         width_ratios=[.2, 1])

axes[0] = plt_preds.box_fullsession_preds(
    y_preds=y_full_pred, session_code=session_code,
    target_EMA_name='LID', ax=axes[0],
    VIOLIN=True, sign_asterix=True,
)

axes[1] = plt_preds.plot_daily_ft_mean(
    y_preds=y_full_pred, y_times=times_full,
    session_code=session_code, ax=axes[1],
    ema_target_name='LID',
)

plt.tight_layout()

if save_plot:
    plt.savefig(os.path.join(lid_preds_figpath, figname), dpi=300)
    print(f'plot saved as {figname} in {lid_preds_figpath}')
    plt.close()

else:
    plt.show()


#### Plot predictions versus l-dopa intake moments


In [ ]:
### subplots per ses

FS=14

fig, axes = plt.subplots(len(test_pred_dict), 1,
                         figsize=(3*len(test_pred_dict), 8),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )


   bp = axes[i_ses].boxplot(box_distance_groups, patch_artist=True,)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

   axes[i_ses].set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
   axes[i_ses].set_xticklabels(dist_labels,)
   axes[i_ses].set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
   axes[i_ses].set_ylim(-2, 9)
   axes[i_ses].set_yticks([1, 3, 5, 7, 9])
   axes[i_ses].set_yticklabels([1, 3, 5, 7, 9])
   axes[i_ses].tick_params(size=FS, labelsize=FS,)

   axes[i_ses].set_title(ses_id, fontsize=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
# plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
#             facecolor='w', dpi=300,)

plt.close()

In [ ]:
### all sessions next to each other

FS=14
pos_adj = [0., .25, .5]

fig, ax = plt.subplots(1, 1, figsize=(9, 3),)

for i_ses, ses_id in enumerate(test_pred_dict):

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]


   pred_dopa_dist = data_handling.get_intervals_to_ldopa(test_stamps, sub=sub_id, ses=ses_id,)
   (box_distance_groups,
    dist_labels) = data_handling.sort_values_into_ldopa_distances(
      values=test_pred, dopa_distances=pred_dopa_dist,
   )

   bp = ax.boxplot(
      box_distance_groups, patch_artist=True, label=ses_id,
      positions=np.arange(len(box_distance_groups)) + pos_adj[i_ses],
      widths=.25,
   )

   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.set_xlabel('Time vs L-Dopa Intake (minutes)', fontsize=FS,)
ax.set_xticks(np.arange(len(dist_labels))+pos_adj[1],)
ax.set_xticklabels(dist_labels,)
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.set_ylim(-3, 9)
ax.set_yticks([-3, -1, 1, 3, 5, 7, 9])
ax.set_yticklabels([-3, -1, 1, 3, 5, 7, 9])
ax.tick_params(size=FS, labelsize=FS,)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))

plt.tight_layout()

figname = 'lidPred_hm24_allSess_ldopaIntakeTime'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()

Daytime blocks

In [ ]:
DAY_HOUR_BLOCKS = {'morning': [8, 12],
                   'afternoon1': [12, 15],
                   'afternoon2': [15, 18], 
                   'evening': [18, 22]}

box_lists = {k: {k2: [] for k2 in DAY_HOUR_BLOCKS}
             for k in test_pred_dict}


for ses_id in test_pred_dict.keys():

   test_stamps = test_stamps_dict[ses_id]
   test_pred = test_pred_dict[ses_id]
   stamp_hrs = [data_handling.get_dayminutes(t)/60 for t in test_stamps]
   
   for t_hr, v in zip(stamp_hrs, test_pred):
      if t_hr < DAY_HOUR_BLOCKS['morning'][1]:
         box_lists[ses_id]['morning'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon1'][1]:
         box_lists[ses_id]['afternoon1'].append(v)
      elif t_hr < DAY_HOUR_BLOCKS['afternoon2'][1]:
         box_lists[ses_id]['afternoon2'].append(v)
      else:
         box_lists[ses_id]['evening'].append(v)

boxparams = {
   'ses01': {'widths': .2, 'positions': np.arange(.0, len(DAY_HOUR_BLOCKS)),},
   'ses02': {'widths': .2, 'positions': np.arange(.2, len(DAY_HOUR_BLOCKS)),},
   'ses03': {'widths': .2, 'positions': np.arange(.4, len(DAY_HOUR_BLOCKS)),}
}
FS=14

fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for i_ses, ses_id in enumerate(box_lists):
   ses_lists = [l for l in box_lists[ses_id].values()]
   bp = ax.boxplot(ses_lists, **boxparams[ses_id],
                   patch_artist=True, label=ses_id)
   for patch in bp['boxes']:
      patch.set_facecolor(SES_COLORS[ses_id])
      patch.set_alpha(.4)

ax.legend(ncol=3, frameon=False, fontsize=FS,
          loc='lower center', bbox_to_anchor=(.5, 1))
ax.set_xticks(list(boxparams.values())[1]['positions'])
ax.set_xticklabels(list(DAY_HOUR_BLOCKS.keys()))
ax.set_ylabel('Dyskinesia prediction\n(EMA scale)', fontsize=FS,)
ax.tick_params(labelsize=FS, size=FS,)

plt.tight_layout()

figname = 'lidPred_hm24_allSess_daytimeBlocks'
plt.savefig(os.path.join(ntrl_fig_path, 'proof_kin_pred', figname),
            facecolor='w', dpi=300,)

plt.close()